In [2]:
!pip install fuzzywuzzy


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import openpyxl as xl 
import tqdm
import os

from fuzzywuzzy import process
import pandas as pd
from openpyxl import load_workbook


C:\Users\hassa\AppData\Local\Programs\Python\Python310\lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
# Get today's date 
from datetime import date
today = date.today()
municipality = 'EMBARK'
template_file = 'details_project_od_excel_CLEAN.xlsx'
target_file = 'details_project_od_excel.xlsx'
routes_file = 'EMBARK_ROUTES.xlsx'
output_file = f'details_project_od_excel_{municipality}_{today}.xlsx'


In [5]:
routes_df = pd.read_excel(routes_file, sheet_name='ROUTES')
# if 'route_short_name' is an empty column, then do the following:

# Change all column names to lowercase

try:
    xfers_df = pd.read_excel(routes_file, sheet_name='XFER')
except:
    # Step 1: Copy routes_df to xfers_df
    xfers_df = routes_df.copy()

    # Step 2: Modify ETC_ROUTE_ID by removing suffix
    xfers_df['ETC_ROUTE_ID'] = xfers_df['ETC_ROUTE_ID'].str.replace(r'(_00|_01)$', '', regex=True)

    # Step 3: Modify ETC_ROUTE_NAME by keeping only the first number
    xfers_df['ETC_ROUTE_NAME'] = xfers_df['ETC_ROUTE_NAME'].str.extract(r'(\d+)')[0]

    # Step 4: Remove duplicates
    xfers_df.drop_duplicates(subset=['ETC_ROUTE_ID', 'ETC_ROUTE_NAME'], inplace=True)

# Change all column names to lowercase
routes_df.columns = [x.lower() for x in routes_df.columns]
xfers_df.columns = [x.lower() for x in xfers_df.columns]

xfers_df_check = xfers_df.copy()
xfers_df['direction_id'] = 0
xfers_df['DIRECTION'] = 'INBOUND'


FileNotFoundError: [Errno 2] No such file or directory: 'EMBARK_ROUTES.xlsx'

In [18]:
def create_empty_excel_from_template(template_file, output_file):
    print('Creating empty Excel file from template: ' + template_file)
    print('template_file: ' + template_file)
    with pd.ExcelFile(template_file, engine='openpyxl') as xls:
        writer = pd.ExcelWriter(output_file, engine='xlsxwriter')

        for sheet in xls.sheet_names:
            df = pd.read_excel(xls, sheet)
            # Set column names to lowercase
            df.columns = map(str.lower, df.columns)


            # Set all values to an empty string
            for col in df.columns:
                df[col] = ''
            df.to_excel(writer, sheet_name=sheet, index=False)

        writer.close()

def fill_excel_sheet(df, workbook_path, sheet_name):
    # Load workbook and select sheet
    book = load_workbook(workbook_path)
    writer = pd.ExcelWriter(workbook_path, engine='openpyxl') 
    writer.book = book
    sheet = book[sheet_name]
    
    # Get headers from Excel sheet
    excel_headers = [cell.value for cell in sheet[1]]
    
    # Get headers from DataFrame
    print(df.head())
    df_headers = df.columns.tolist()
    
    # Match DataFrame headers with Excel headers using fuzzy matching
    header_mapping = {}
    for e_header in excel_headers:
        if e_header is not None:  # Skip empty headers in Excel
            closest_match, score = process.extractOne(e_header, df_headers)
            
            # If similarity score is above a threshold (e.g., 80), consider them a match
            if score > 80:
                header_mapping[e_header] = closest_match
                
    # Update Excel sheet with DataFrame values for matching headers
    for e_col, df_col in header_mapping.items():
        excel_col_idx = excel_headers.index(e_col) + 1  # Excel is 1-indexed
        for row_idx, value in enumerate(df[df_col], 2):  # Skip header, start at row 2
            sheet.cell(row=row_idx, column=excel_col_idx, value=value)
            
    # Save the updated workbook
    writer.close()

def fill_from_additional_files(template_file, additional_files):
    print('Filling empty Excel file from additional files: ' + str(additional_files))
    # Load the empty (or template) Excel
    with pd.ExcelFile(template_file, engine='openpyxl') as empty_xls:
        writer = pd.ExcelWriter(template_file, engine='xlsxwriter')

        for sheet in empty_xls.sheet_names:
            try:
                empty_df = pd.read_excel(empty_xls, sheet)
            except:
                empty_df = pd.read_excel(empty_xls, sheet, engine='openpyxl', header=None)

            # Check each additional file for matching sheets and columns
            for add_file in additional_files:
                with pd.ExcelFile(add_file, engine='openpyxl') as add_xls:
                    if sheet in add_xls.sheet_names:
                        add_df = pd.read_excel(add_xls, sheet)

                        for col in add_df.columns:
                            if col in empty_df.columns:
                                empty_df[col] = add_df[col]

            # Write the updated data back to the empty/template Excel file
            empty_df.to_excel(writer, sheet_name=sheet, index=False)

        writer.save()


def populate_from_template(template_file, additional_files, output_file):
    dfs = {}  # To store empty DataFrames with structure from the template
    
    # Create empty DataFrame for each sheet in the template
    with pd.ExcelFile(template_file, engine='openpyxl') as template_xls:
        for sheet in template_xls.sheet_names:
            df = pd.read_excel(template_xls, sheet)
            for col in df.columns:
                df[col] = ''
            dfs[sheet] = df
    
    # Populate data from additional files
    for add_file in additional_files:
        with pd.ExcelFile(add_file, engine='openpyxl') as add_xls:
            for sheet in add_xls.sheet_names:
                if sheet in dfs:
                    add_df = pd.read_excel(add_xls, sheet)
                    for col in add_df.columns:
                        if col in dfs[sheet].columns:
                            dfs[sheet][col] = add_df[col]

    # Write the populated DataFrames to the output file
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        for sheet, df in dfs.items():
            df.to_excel(writer, sheet_name=sheet, index=False)

def read_stop_files():
    # Initialize empty dataframes for each sheet
    stops_v1_df = pd.DataFrame()
    stops_df = pd.DataFrame()
    xfer_stops_df = pd.DataFrame()
    
    # Iterate through all files in the directory
    for filename in os.listdir():
        print(filename)
        # Check if the filename contains "STOP" and ends with ".xlsx"
        if ("STOPS" in filename or 'stops' in filename) and filename.endswith(".xlsx"):
            # filepath = os.path.join(directory_path, filename)
            print('Reading file: ' + filename)
            # Read each sheet from the Excel file into a dataframe using sheet index (0, 1, 2)
            temp_stops_v1_df = pd.read_excel(filename, sheet_name=0)
            try:
                temp_stops_df = pd.read_excel(filename, sheet_name='STOPS')
                seq_fixed_column = temp_stops_df.columns[8]
                temp_stops_df = temp_stops_df.rename(columns={seq_fixed_column: 'seq_fixed'})
            except:
                print('No sheet 1')
                temp_stops_df = pd.DataFrame()
            try:
                temp_xfer_stops_df = pd.read_excel(filename, sheet_name='XFER-STOPS')
                seq_fixed_column = temp_xfer_stops_df.columns[8]
                temp_xfer_stops_df = temp_xfer_stops_df.rename(columns={seq_fixed_column: 'seq_fixed'})
            except:
                print('No sheet 2')
                temp_xfer_stops_df = pd.DataFrame()
            # Append the data to the respective dataframes
            stops_v1_df = pd.concat([stops_v1_df, temp_stops_v1_df])
            stops_df = temp_stops_v1_df#stops_v1_df#pd.concat([stops_df, temp_stops_df])
            xfer_stops_df = pd.concat([xfer_stops_df, temp_xfer_stops_df])
        # Set stops_df["etc_route_id"] equal to agency + _ + gtfs_ver + _ + route_short_name _ + 0 + direction_id
    #print(stops_df.columns)

    # Convert stops_df column names to lowercase
    stops_df.columns = map(str.lower, stops_df.columns)
    print("STOPS df columns: ")
    print(stops_df.columns)
    # Convert xfer_stops_df column names to lowercase
    xfer_stops_df.columns = map(str.lower, xfer_stops_df.columns)

    stops_df['agency'] = stops_df['agency'].astype(str)
    stops_df['gtfs_ver'] = stops_df['gtfs_ver'].astype(str)
    stops_df['route_short_name'] = stops_df['route_short_name'].astype(str)
    stops_df['direction_id'] = stops_df['direction_id'].astype(str)

    # Create the new 'etc_route_id' column by concatenating the string columns
    stops_df['etc_route_id'] = stops_df['agency'] + '_' + stops_df['gtfs_ver'] + '_' + stops_df['route_short_name'] + '_0' + stops_df['direction_id']
    # Set stops_df["etc_route_name"] equal to routes_df["route_name"] where stops_df["etc_route_id"] == routes_df["route_id"]
    # Create a dictionary from routes_df for the mapping
    route_id_to_name = dict(zip(routes_df['etc_route_id'], routes_df['etc_route_name']))

    # Use map to populate etc_route_name based on etc_route_id
    stops_df['etc_route_name'] = stops_df['etc_route_id'].map(route_id_to_name)
    
            
    return stops_v1_df, stops_df, xfer_stops_df

def add_columns_to_xfers(xfers_df, stops_df):
    # Convert stops_df column names to lowercase
    stops_df.columns = map(str.lower, stops_df.columns)
    stops_df['etc_route_id'] = stops_df['etc_route_id'].str.rstrip('_00')
    # Get the list of columns from stops_df
    columns = stops_df.columns.tolist()
    
    # Get unique 'etc_route_id' values from xfers_df
    unique_route_names = xfers_df['etc_route_id'].dropna().unique()
    
    # Iterate through unique 'etc_route_id' values and find matching rows in stops_df
    for route_name in unique_route_names:
        matching_rows = stops_df[stops_df['etc_route_id'] == route_name]
        # Take the first matching row, if available
        matching_row = matching_rows.iloc[0] if not matching_rows.empty else None

        # If a matching row is found in stops_df, update all corresponding rows in xfers_df
        if matching_row is not None:
            for col in columns:
                xfers_df.loc[xfers_df['etc_route_id'] == route_name, col] = matching_row[col.lower()]
    
    return xfers_df
def create_sis_route_df(df):
    # Initialize an empty DataFrame to store the final output
    sis_route_df = pd.DataFrame(columns=['etc_route_id', 'etc_route_name', 'sis_etc_route_id', 'sis_etc_route_name'])
    
    # Create a dictionary to keep track of seen prefixes
    seen = {}
    
    for idx, row in df.iterrows():
        prefix = row['etc_route_id'][:-2]
        
        if prefix in seen:
            # Prepare a new row for the DataFrame by merging the current row and the seen row
            reversible_row1 = {
                'etc_route_id': row['etc_route_id'],
                'etc_route_name': row['etc_route_name'],
                'sis_etc_route_id': seen[prefix]['etc_route_id'],
                'sis_etc_route_name': seen[prefix]['etc_route_name']
            }
            reversible_row2 = {
                'etc_route_id': seen[prefix]['etc_route_id'],
                'etc_route_name': seen[prefix]['etc_route_name'],
                'sis_etc_route_id': row['etc_route_id'],
                'sis_etc_route_name': row['etc_route_name']
            }
            # Append the new rows to the DataFrame
            sis_route_df = sis_route_df.append(reversible_row1, ignore_index=True)
            sis_route_df = sis_route_df.append(reversible_row2, ignore_index=True)
            
            # Remove the prefix from seen as we've processed the pair
            del seen[prefix]
        else:
            seen[prefix] = row
    
    # Handle non-reversible rows
    for prefix, row in seen.items():
        non_reversible_row = {
            'etc_route_id': row['etc_route_id'],
            'etc_route_name': row['etc_route_name'],
            'sis_etc_route_id': None,
            'sis_etc_route_name': None
        }
        sis_route_df = sis_route_df.append(non_reversible_row, ignore_index=True)
    #count the number of times a sis_route_df['etc_route_id'] appears in stops_df['etc_route_id']
    sis_route_df['num_of_stops'] = sis_route_df['etc_route_id'].map(stops_df['etc_route_id'].value_counts())
        
    return sis_route_df




#directory_path = municipality


# Example usage:
def transform_text(text):
    return '_'.join(text.split('_')[:-1])
# output_file = 'empty_details_project.xlsx'
# additional_files = ['AIR-UNI-K12_VALLEY_METRO_PLACENAMES_20230203.xlsx', '2023 VM _ OD Sampling Plan _ 2-14-23.xlsx', 'ValleyMetro_ROUTES_JAN24.xlsx', 'ValleyMetro_STOPS_V1_seq_fixed.xlsx']
create_empty_excel_from_template(template_file, output_file)
stops_v1_df, stops_df, xfer_stops_df = read_stop_files()
stops_df['route_dir'] = stops_df['route_short_name'] + '_' + stops_df['direction_id'].astype(str)
fill_excel_sheet(stops_df, output_file, 'STOPS')
xfers_routes_df = pd.read_excel(routes_file, sheet_name='XFERS-ROUTES')
#xfers_etc_route_id = xfer_stops_df['etc_route_id']
xfer_stops_df.columns = [x.lower() for x in xfer_stops_df.columns]

# Create a column called   etc_route_id_helper that is the ETC_ROUTE_ID without the suffix after the last _
print("Working on XFERS_ROUTES_DF")
xfers_routes_df['etc_route_id'] = xfers_df['etc_route_id']
xfers_routes_df['etc_route_name'] = xfers_df['etc_route_name']
xfers_routes_df['etc_route_id'] = xfers_routes_df['etc_route_id'].str.replace(r'\s*\([^)]*\)', '', regex=True)
xfers_routes_df['etc_route_id_helper'] = xfers_routes_df['etc_route_id']
xfer_stops_df['etc_route_id_helper'] = xfer_stops_df['etc_stop_id'].apply(transform_text)
# if   etc_route_id_helper matches  etc_route_id_helper in xfers_df, then for the empty columns in routes_df, fill them with the corresponding values in xfers_df
# Get the list of columns from xfers_df
columns = xfer_stops_df.columns.tolist()
# Get unique 'etc_route_id' values from routes_df
unique_route_names = xfers_routes_df['etc_route_id_helper'].dropna().unique()
# Iterate through unique 'etc_route_id' values and find matching rows in xfers_df
for route_name in unique_route_names:
    print(route_name)
    print(xfer_stops_df['etc_route_id_helper'])
    matching_rows = xfer_stops_df[xfer_stops_df['etc_route_id_helper'] == route_name]
    # Take the first matching row, if available
    matching_row = matching_rows.iloc[0] if not matching_rows.empty else None
    # If a matching row is found in xfers_df, update all corresponding rows in routes_df
    if matching_row is not None:
        for col in columns:
            print(col)
            xfers_routes_df.loc[xfers_routes_df['etc_route_id_helper'] == route_name, col] = matching_row[col]
# Remove the   etc_route_id_helper column

# Add the columns from stops_df to xfers_df


fill_excel_sheet(xfers_routes_df, output_file, 'XFERS')
#filled_xfers = add_columns_to_xfers(xfers_df, stops_df)
#fill_excel_sheet(filled_xfers, output_file, 'XFERS')
sis_route_df = create_sis_route_df(routes_df)
fill_excel_sheet(sis_route_df, output_file, 'SIS_ROUTE')



# # fill_from_additional_files(template_file, additional_files)
# populate_from_template(template_file, additional_files, output_file)

Creating empty Excel file from template: details_project_od_excel_CLEAN.xlsx
template_file: details_project_od_excel_CLEAN.xlsx
combined_routes.csv
project_CR_CLEAN.xlsx
20240325EMBARK_OD_FINAL Sampling Plan_csv.csv
EMBARK_STOPS_SEQUENCED_V1 (1).xlsx
Reading file: EMBARK_STOPS_SEQUENCED_V1 (1).xlsx
details_project_od_excel_EMBARK_2024-05-21.xlsx
.~lock.EMBARK_ROUTES.xlsx#
concatenated_df_preview_ST T Line.csv
project_CR_SEATTLE.xlsx
project_CR_EMBARK copy.xlsx
details_project_od_excel_UTA.xlsx
project_CR_EMBARK.xlsx
TOD.xlsx
.~lock.20240325EMBARK_OD_FINAL Sampling Plan.xlsx#
details_project_od_excel_EMBARK.xlsx
20240325EMBARK_OD_FINAL Sampling Plan.xlsx
rail_cr.xlsx
details_project_od_excel_SEATTLE_2024-04-18.xlsx
.~lock.EMBARK_STOPS_SEQUENCED_V1 (1).xlsx#
project_CR_CLEAN_template.xlsx
concatenated_df_preview_ST One Line.csv
details_project_od_excel_CLEAN.xlsx
rtmatch.csv
concat_CR.csv
updated_matched_routes.csv
route_total2.csv
details_project_od_Seattle.xlsx
EMBARK_ROUTES.xlsx
detai

/tmp/ipykernel_78480/4145522414.py:24: FutureWarning: Setting the `book` attribute is not part of the public API, usage can give unexpected or corrupted results and will be removed in a future version
  writer.book = book


Working on XFERS_ROUTES_DF
EMB_1_002
0         EMB_1_002
1         EMB_1_002
2         EMB_1_002
3         EMB_1_002
4         EMB_1_002
           ...     
1957    EMB_1_RAPID
1958    EMB_1_RAPID
1959    EMB_1_RAPID
1960    EMB_1_RAPID
1961    EMB_1_RAPID
Name: etc_route_id_helper, Length: 1962, dtype: object
id
longitude
latitude
agency
gtfs_ver
gtfs_date
route_short_name
short_clean
seq_fixed


/tmp/ipykernel_78480/4145522414.py:278: FutureWarning: reindexing with a non-unique Index is deprecated and will raise in a future version.
  xfers_routes_df.loc[xfers_routes_df['etc_route_id_helper'] == route_name, col] = matching_row[col]


ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
xfers = xfers_routes_df[['agency', 'gtfs_ver', 'gtfs_date', 'route_short_name', 'route_long_name', 'direction_id', 'direction', 'etc_route_id']]
xfers['etc_route_name'] = xfers_df['etc_route_name']
stops_df['route_long_name'] = stops_df['route_long_name'].fillna('NO ROUTE LONG NAME')
for index, xfer_row in xfers.iterrows():
    # For each row in xfers, check against every route in stops_df
    for stop_row in stops_df.itertuples():
        print("RLN: ",stop_row.route_long_name)
        # Check if the route_long_name from stops_df is a substring of etc_route_name in xfers
        if stop_row.route_long_name in xfer_row.etc_route_name:
            # If a match is found, assign the route_long_name to xfers' corresponding column
            xfers.at[index, 'route_long_name'] = stop_row.route_long_name
            # Optionally break the loop if only the first match is needed
            break
xfers.to_excel(f"{municipality}_XFERS_details.xlsx")

KeyError: "['route_long_name', 'direction'] not in index"

In [ ]:
def populate_from_template_with_aliases(template_file, additional_files, output_file):
    dfs = {}  # To store empty DataFrames with structure from the template
    
    # Create empty DataFrame for each sheet in the template
    with pd.ExcelFile(template_file, engine='openpyxl') as template_xls:
        for sheet in template_xls.sheet_names:
            df = pd.read_excel(template_xls, sheet)
            for col in df.columns:
                df[col] = ''
            dfs[sheet] = df
    
    # Populate data from additional files
    for add_file in additional_files:
        with pd.ExcelFile(add_file, engine='openpyxl') as add_xls:
            for sheet in add_xls.sheet_names:
                if sheet in dfs:
                    add_df = pd.read_excel(add_xls, sheet)
                    for col in add_df.columns:
                        if col in dfs[sheet].columns:
                            dfs[sheet][col] = add_df[col]

    # Write the populated DataFrames to the output file
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        for sheet, df in dfs.items():
            df.to_excel(writer, sheet_name=sheet, index=False)

# populate_from_template_with_aliases(template_file, additional_files, output_file)                                                                                                                                                                             

In [ ]:

# # Specify the file path of the details file created today
# details_file_path = "details_project_od_excel_oki_2024-01-29.xlsx"

# # Read the STOPS sheet of the details file into a DataFrame
# df = pd.read_excel(details_file_path, sheet_name="STOPS")

# # Find duplicate rows based on all columns
# duplicate_rows = df[df.duplicated(keep=False)]

# # Print the duplicate rows
# print(duplicate_rows)


In [ ]:
# import pandas as pd

# # Read the "STOPS" sheet of the Excel file
# df_stops = pd.read_excel('BCRTA_STOPS_V1_SEQUENCED.xlsx', sheet_name='STOPS')

# # Extract the ETC_ROUTE_ID prefix from ETC_STOP_ID
# df_stops['etc_route_id_prefix'] = df_stops['ETC_STOP_ID'].str.rsplit('_', n=1).str[0]

# # Merge df_stops with routes_df on ETC_ROUTE_ID_PREFIX
# df_stops = df_stops.merge(routes_df[['etc_route_id', 'etc_route_name']], left_on='etc_route_id_prefix', right_on='etc_route_id', how='left')

# # Fill the missing values in route_long_name with ETC_ROUTE_NAME
# df_stops['route_long_name'] = df_stops['route_long_name'].fillna(df_stops['etc_route_name'])

# # Drop the unnecessary columns
# df_stops = df_stops.drop(['etc_route_id_prefix', 'etc_route_id', 'etc_route_name'], axis=1)

# # For df_stops['route_long_name'], remove everything after the first hyphen
# df_stops['route_long_name'] = df_stops['route_long_name'].str.rsplit('-', n=1).str[0]


# # Get unique combinations of "LAT6" and "StopName"
# unique_combinations = df_stops.drop_duplicates(subset=['LAT6', 'StopName'])
# unique_combinations.to_excel('unique_combinations.xlsx', index=False)
# # Print the unique combinations
# print(unique_combinations)
